# 🌳 Minería de Datos — Árboles de Decisión con Dataset Titanic
## Clase completa de 2 horas + taller final

**Asignatura:** Minería de Datos  
**Tema:** Clasificación supervisada con árboles de decisión  
**Duración estimada:** 2 horas  
**Nivel:** Ingeniería de Sistemas  

---

## Propósito de la clase

En esta clase estudiaremos **árboles de decisión** usando un dataset en línea estable y ampliamente usado en ciencia de datos: **Titanic**.

El problema consiste en predecir si un pasajero sobrevivió o no al naufragio a partir de variables como:

- clase del pasajero,
- sexo,
- edad,
- tarifa pagada,
- número de familiares a bordo.

Este dataset es ideal para clase porque permite explicar:

1. Carga de datos desde internet.
2. Comprensión del problema.
3. Limpieza básica de datos.
4. Conversión de variables categóricas.
5. Separación train/test.
6. Entrenamiento de árboles de decisión.
7. Evaluación con métricas.
8. Interpretación de reglas.
9. Overfitting por profundidad.
10. Taller aplicado al final.

---

## Pregunta de minería de datos

> ¿Podemos predecir si un pasajero sobrevivió al Titanic usando sus características registradas?

Formalmente:

\[
f(X) \rightarrow y
\]

Donde:

- \(X\): características del pasajero.
- \(y\): sobrevivió o no sobrevivió.
- \(f\): modelo de clasificación aprendido desde los datos.

# 1. Ubicación dentro del curso

Hasta este punto del curso ya se han trabajado:

- Conceptos generales de minería de datos.
- CRISP-DM.
- Preparación de datos.
- Train/Test.
- Métricas de clasificación.
- Regresión logística.

Ahora se introduce un modelo supervisado distinto: **árboles de decisión**.

---

## ¿Por qué árboles de decisión?

Porque combinan dos elementos muy importantes:

1. **Capacidad predictiva:** permiten clasificar nuevos registros.
2. **Interpretabilidad:** generan reglas fáciles de explicar.

Mientras otros modelos pueden verse como “cajas negras”, un árbol permite explicar decisiones mediante reglas:

```text
SI Sex = female
Y Pclass <= 2
ENTONCES mayor probabilidad de sobrevivir
```

Esto conecta directamente con minería de datos aplicada: no basta predecir, también hay que explicar.

# 2. Concepto de árbol de decisión

Un árbol de decisión es un modelo que clasifica observaciones haciendo preguntas sucesivas.

Cada pregunta divide los datos en subconjuntos más homogéneos.

---

## Estructura de un árbol

| Elemento | Significado |
|---|---|
| Nodo raíz | Primera pregunta del árbol |
| Nodo interno | Pregunta intermedia |
| Rama | Resultado de una pregunta |
| Hoja | Decisión final |
| Profundidad | Número de niveles del árbol |
| Regla | Camino desde la raíz hasta una hoja |

---

## Ejemplo intuitivo

```text
¿El pasajero era mujer?
├── Sí  → Mayor probabilidad de sobrevivir
└── No
    ├── ¿Era niño?
    │   ├── Sí → Probabilidad intermedia
    │   └── No → Menor probabilidad
```

El árbol aprende estas reglas desde los datos.

# 3. Dataset Titanic

Usaremos un CSV público alojado en GitHub:

```python
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
```

Este dataset contiene información de pasajeros del Titanic.

---

## Algunas variables importantes

| Variable | Significado |
|---|---|
| PassengerId | Identificador del pasajero |
| Survived | 0 = no sobrevivió, 1 = sobrevivió |
| Pclass | Clase del tiquete: 1, 2 o 3 |
| Name | Nombre |
| Sex | Sexo |
| Age | Edad |
| SibSp | Hermanos/cónyuges a bordo |
| Parch | Padres/hijos a bordo |
| Ticket | Número de tiquete |
| Fare | Tarifa pagada |
| Cabin | Cabina |
| Embarked | Puerto de embarque |

---

## Variable objetivo

La variable que queremos predecir es:

\[
Survived
\]

- 0: No sobrevivió.
- 1: Sobrevivió.

In [ ]:
# ============================================================
# 1. Importar librerías
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 120)

# 4. Carga del dataset desde internet

En proyectos reales los datos pueden venir de:

- archivos CSV,
- bases de datos,
- APIs,
- repositorios públicos,
- sistemas transaccionales.

Aquí usamos un CSV en línea.

Si el profesor quiere blindar la clase, puede descargar el CSV antes de la sesión y tenerlo como respaldo.

In [ ]:
# ============================================================
# 2. Cargar dataset Titanic desde URL estable
# ============================================================

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)

print("Dataset cargado correctamente.")
print("Dimensiones:", df.shape)

df.head()

# 5. Comprensión inicial del dataset

Antes de modelar debemos responder:

1. ¿Cuántas filas y columnas tiene el dataset?
2. ¿Qué tipos de variables hay?
3. ¿Cuál es la variable objetivo?
4. ¿Hay valores faltantes?
5. ¿Hay columnas que no aportan al modelo?
6. ¿Existen variables categóricas que deben codificarse?

Este paso corresponde a la fase de **comprensión de datos** del proceso CRISP-DM.

In [ ]:
# Información general del dataset
df.info()

In [ ]:
# Resumen estadístico básico
df.describe(include="all").T

In [ ]:
# Valores faltantes por columna
faltantes = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
}).sort_values("faltantes", ascending=False)

faltantes

## Interpretación de valores faltantes

En Titanic suelen aparecer valores faltantes en:

- `Age`: edad no registrada para algunos pasajeros.
- `Cabin`: mucha información faltante.
- `Embarked`: algunos pocos valores faltantes.

Para esta clase:

- Usaremos `Age`, imputando los faltantes con la mediana.
- No usaremos `Cabin`, porque tiene demasiados faltantes.
- Usaremos `Embarked`, imputando con la moda si es necesario.

# 6. Distribución de la variable objetivo

Revisaremos cuántos pasajeros sobrevivieron y cuántos no.

Esto es importante porque una clasificación con clases desbalanceadas puede hacer que el accuracy sea engañoso.

In [ ]:
conteo = df["Survived"].value_counts().sort_index()
porcentaje = df["Survived"].value_counts(normalize=True).sort_index() * 100

resumen_target = pd.DataFrame({
    "Survived": conteo.index,
    "descripcion": ["No sobrevivió", "Sobrevivió"],
    "conteo": conteo.values,
    "porcentaje": porcentaje.round(2).values
})

resumen_target

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(resumen_target["descripcion"], resumen_target["conteo"])
plt.title("Distribución de supervivencia")
plt.xlabel("Clase")
plt.ylabel("Número de pasajeros")
plt.show()

## Preguntas para clase

1. ¿Hay más sobrevivientes o no sobrevivientes?
2. ¿El dataset está muy desbalanceado?
3. ¿Qué pasaría si un modelo predijera siempre “no sobrevivió”?
4. ¿Accuracy sería suficiente?

Esta discusión conecta con matriz de confusión y métricas de clasificación.

# 7. Exploración visual sencilla

Antes de entrenar el árbol, observemos algunas relaciones:

- Supervivencia por sexo.
- Supervivencia por clase.
- Distribución de edad por supervivencia.
- Tarifa pagada por supervivencia.

Esto ayuda a entender qué patrones podría aprender el árbol.

In [ ]:
# Supervivencia por sexo
tabla_sex = pd.crosstab(df["Sex"], df["Survived"], normalize="index") * 100
tabla_sex.columns = ["No sobrevivió (%)", "Sobrevivió (%)"]
tabla_sex.round(2)

In [ ]:
tabla_sex.plot(kind="bar", figsize=(7,4))
plt.title("Supervivencia por sexo")
plt.xlabel("Sexo")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Supervivencia por clase
tabla_clase = pd.crosstab(df["Pclass"], df["Survived"], normalize="index") * 100
tabla_clase.columns = ["No sobrevivió (%)", "Sobrevivió (%)"]
tabla_clase.round(2)

In [ ]:
tabla_clase.plot(kind="bar", figsize=(7,4))
plt.title("Supervivencia por clase del pasajero")
plt.xlabel("Clase")
plt.ylabel("Porcentaje")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Edad por supervivencia
plt.figure(figsize=(8,4))
plt.hist(df[df["Survived"] == 0]["Age"].dropna(), alpha=0.6, label="No sobrevivió")
plt.hist(df[df["Survived"] == 1]["Age"].dropna(), alpha=0.6, label="Sobrevivió")
plt.title("Distribución de edad por supervivencia")
plt.xlabel("Edad")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()

# 8. Preparación de datos

No todas las columnas son útiles directamente para el modelo.

## Columnas que no usaremos

| Columna | Razón |
|---|---|
| PassengerId | Identificador, no aporta patrón general |
| Name | Texto complejo, requiere procesamiento adicional |
| Ticket | Código alfanumérico difícil de usar directamente |
| Cabin | Muchos faltantes |
| Survived | Es la variable objetivo |

## Variables que sí usaremos

- Pclass
- Sex
- Age
- SibSp
- Parch
- Fare
- Embarked

---

## Conversión de variables categóricas

Los árboles de `scikit-learn` requieren variables numéricas.

Convertiremos:

- `Sex`: male/female a 0/1.
- `Embarked`: variables dummy.

In [ ]:
# ============================================================
# 3. Preparación del dataset para modelado
# ============================================================

df_model = df[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

# Imputar edad con mediana
df_model["Age"] = df_model["Age"].fillna(df_model["Age"].median())

# Imputar puerto de embarque con moda
df_model["Embarked"] = df_model["Embarked"].fillna(df_model["Embarked"].mode()[0])

# Convertir Sex a variable numérica
df_model["Sex"] = df_model["Sex"].map({"male": 0, "female": 1})

# Convertir Embarked en variables dummy
df_model = pd.get_dummies(df_model, columns=["Embarked"], drop_first=True)

df_model.head()

In [ ]:
# Verificación final
print("Dimensiones del dataset modelado:", df_model.shape)
print("\nValores faltantes después de limpieza:")
print(df_model.isna().sum())

# 9. Separación entre X e y

Definimos:

- `X`: variables predictoras.
- `y`: variable objetivo.

La variable `Survived` no debe estar dentro de `X`, porque sería fuga de información.

In [ ]:
X = df_model.drop(columns=["Survived"])
y = df_model["Survived"]

print("X:", X.shape)
print("y:", y.shape)
print("Variables predictoras:")
print(list(X.columns))

# 10. Train/Test split

Separaremos:

- 70% para entrenamiento.
- 30% para prueba.

Usamos `stratify=y` para conservar proporciones similares de sobrevivientes y no sobrevivientes en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nDistribución train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nDistribución test:")
print(y_test.value_counts(normalize=True).sort_index())

# 11. ¿Cómo aprende un árbol?

El árbol busca divisiones del tipo:

\[
x_j \leq c
\]

Donde:

- \(x_j\) es una variable.
- \(c\) es un punto de corte.

Ejemplo:

```text
Sex <= 0.5
Fare <= 26.25
Age <= 12.5
```

Cada división busca que los grupos resultantes sean más puros.

# 12. Impureza, Gini y Entropía

Un nodo es puro si contiene observaciones de una sola clase.

Ejemplo:

```text
Nodo A: 40 no sobrevivieron, 0 sobrevivieron → puro
Nodo B: 20 no sobrevivieron, 20 sobrevivieron → impuro
```

---

## Índice de Gini

\[
Gini = 1 - \sum_{i=1}^{k} p_i^2
\]

## Entropía

\[
Entropy = - \sum_{i=1}^{k} p_i \log_2(p_i)
\]

Ambas medidas buscan cuantificar qué tan mezcladas están las clases en un nodo.

In [ ]:
def gini(proporciones):
    return 1 - sum([p**2 for p in proporciones])

def entropia(proporciones):
    resultado = 0
    for p in proporciones:
        if p > 0:
            resultado -= p * np.log2(p)
    return resultado

ejemplos = {
    "Nodo puro [1.0, 0.0]": [1.0, 0.0],
    "Nodo 50/50 [0.5, 0.5]": [0.5, 0.5],
    "Nodo 80/20 [0.8, 0.2]": [0.8, 0.2],
    "Nodo 90/10 [0.9, 0.1]": [0.9, 0.1]
}

for nombre, props in ejemplos.items():
    print(nombre, "| Gini:", round(gini(props), 3), "| Entropía:", round(entropia(props), 3))

# 13. Entrenamiento del árbol base

Entrenaremos un árbol con:

- `criterion="gini"`
- `max_depth=3`
- `random_state=42`

Limitamos la profundidad para que el árbol sea interpretable.

In [ ]:
arbol_base = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

arbol_base.fit(X_train, y_train)

print("Árbol base entrenado correctamente.")

# 14. Evaluación del modelo

Evaluaremos:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusión

Recordemos:

\[
Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
\]

\[
Precision = \frac{TP}{TP + FP}
\]

\[
Recall = \frac{TP}{TP + FN}
\]

\[
F1 = 2\frac{Precision \cdot Recall}{Precision + Recall}
\]

In [ ]:
y_pred = arbol_base.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1-score:", round(f1_score(y_test, y_pred), 4))

print("\nReporte completo:")
print(classification_report(y_test, y_pred, target_names=["No sobrevivió", "Sobrevivió"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No sobrevivió", "Sobrevivió"]
)

disp.plot()
plt.title("Matriz de confusión - Árbol base")
plt.show()

# 15. Interpretación de la matriz de confusión

En este problema:

- Clase 0: no sobrevivió.
- Clase 1: sobrevivió.

La matriz permite ver:

| Tipo de resultado | Interpretación |
|---|---|
| Verdadero negativo | Predijo no sobrevivió y realmente no sobrevivió |
| Verdadero positivo | Predijo sobrevivió y realmente sobrevivió |
| Falso positivo | Predijo sobrevivió pero no sobrevivió |
| Falso negativo | Predijo no sobrevivió pero sí sobrevivió |

---

## Pregunta crítica

¿Qué tipo de error parece más grave en este contexto histórico?

No es un problema médico actual, pero sirve para discutir cómo el costo del error depende del contexto.

# 16. Visualización del árbol

Ahora visualizaremos el árbol.

Cada nodo muestra:

- Condición de división.
- Índice de Gini.
- Número de muestras.
- Distribución de clases.
- Clase predicha.

In [ ]:
plt.figure(figsize=(24, 12))

plot_tree(
    arbol_base,
    feature_names=X.columns,
    class_names=["No sobrevivió", "Sobrevivió"],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Árbol de decisión - Titanic")
plt.show()

# 17. Reglas del árbol en texto

Exportar el árbol como texto permite leer las reglas más claramente.

Esto es útil para transformar el modelo en explicación.

In [ ]:
reglas = export_text(arbol_base, feature_names=list(X.columns))
print(reglas)

## Actividad rápida

Seleccione una regla y escríbala en lenguaje natural.

Ejemplo:

```text
Si Sex <= 0.5 y Pclass > 1.5, entonces el modelo predice no sobrevivió.
```

Luego responda:

1. ¿Qué significa `Sex <= 0.5`?
2. ¿Qué significa `Pclass`?
3. ¿La regla parece tener sentido histórico?

# 18. Importancia de variables

Los árboles permiten calcular importancia de variables.

Esto indica qué variables ayudaron más a reducir impureza.

No significa causalidad, sino utilidad predictiva dentro del modelo.

In [ ]:
importancias = pd.DataFrame({
    "variable": X.columns,
    "importancia": arbol_base.feature_importances_
}).sort_values("importancia", ascending=False)

importancias

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(importancias["variable"], importancias["importancia"])
plt.gca().invert_yaxis()
plt.title("Importancia de variables según el árbol")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.show()

# 19. Overfitting en árboles

Los árboles pueden sobreajustar.

Un árbol muy profundo puede memorizar detalles del entrenamiento:

```text
Si Sex = female
y Age = 28
y Fare = 13.5
y Parch = 1
entonces...
```

Eso puede funcionar muy bien en train, pero mal en test.

---

## Señales de overfitting

| Señal | Interpretación |
|---|---|
| Accuracy train muy alta | El modelo aprendió mucho del entrenamiento |
| Accuracy test menor | Mala generalización |
| Árbol muy profundo | Reglas demasiado específicas |
| Hojas con pocas observaciones | Memorización posible |

# 20. Comparación de profundidades

Entrenaremos árboles con diferentes valores de `max_depth`.

Compararemos:

- Accuracy train
- Accuracy test
- F1 test

Esto ayuda a decidir qué profundidad es razonable.

In [ ]:
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, None]
resultados = []

for p in profundidades:
    modelo = DecisionTreeClassifier(
        criterion="gini",
        max_depth=p,
        random_state=42
    )

    modelo.fit(X_train, y_train)

    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)

    resultados.append({
        "max_depth": str(p),
        "accuracy_train": accuracy_score(y_train, pred_train),
        "accuracy_test": accuracy_score(y_test, pred_test),
        "f1_test": f1_score(y_test, pred_test)
    })

df_resultados = pd.DataFrame(resultados)
df_resultados

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(df_resultados["max_depth"], df_resultados["accuracy_train"], marker="o", label="Train")
plt.plot(df_resultados["max_depth"], df_resultados["accuracy_test"], marker="o", label="Test")

plt.title("Efecto de la profundidad del árbol")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Interpretación esperada

Preguntas para clase:

1. ¿El accuracy de train aumenta con la profundidad?
2. ¿El accuracy de test aumenta siempre?
3. ¿En qué punto aparece posible sobreajuste?
4. ¿Qué profundidad parece más razonable?
5. ¿Cuál árbol sería más fácil de explicar?

# 21. Comparación Gini vs Entropía

Ahora comparamos dos árboles:

- Uno con `criterion="gini"`.
- Uno con `criterion="entropy"`.

Ambos con la misma profundidad.

In [ ]:
comparacion = []

for criterio in ["gini", "entropy"]:
    modelo = DecisionTreeClassifier(
        criterion=criterio,
        max_depth=3,
        random_state=42
    )

    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    comparacion.append({
        "criterio": criterio,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred)
    })

pd.DataFrame(comparacion)

# 22. Regularización del árbol

Además de `max_depth`, se pueden usar otros parámetros:

| Parámetro | Función |
|---|---|
| `max_depth` | Limita profundidad |
| `min_samples_split` | Mínimo de muestras para dividir un nodo |
| `min_samples_leaf` | Mínimo de muestras en una hoja |
| `criterion` | Criterio de división |
| `class_weight` | Peso de clases |

Estos parámetros controlan la complejidad del modelo.

In [ ]:
arbol_regularizado = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_leaf=10,
    random_state=42
)

arbol_regularizado.fit(X_train, y_train)
pred_reg = arbol_regularizado.predict(X_test)

print(classification_report(y_test, pred_reg, target_names=["No sobrevivió", "Sobrevivió"]))

# 23. Lectura crítica

Un árbol de decisión no debe evaluarse solo por accuracy.

Una evaluación profesional considera:

1. Métricas.
2. Matriz de confusión.
3. Interpretación de errores.
4. Interpretabilidad.
5. Posible sobreajuste.
6. Calidad del dataset.
7. Variables excluidas.
8. Sesgos históricos del dato.

---

## Ventajas

- Fácil de explicar.
- No requiere escalado.
- Produce reglas.
- Permite visualizar la decisión.
- Es útil como modelo base.

## Desventajas

- Puede sobreajustar.
- Puede ser inestable.
- Depende de la calidad de los datos.
- Puede reflejar sesgos del dataset.
- Un solo árbol no siempre es el mejor predictor.

# 🧪 24. Taller final de clase

## Objetivo

Entrenar, evaluar e interpretar un árbol de decisión usando el dataset Titanic.

## Forma de trabajo

Grupos de 2 o 3 estudiantes.

## Entregable

Un notebook o documento con:

1. Análisis inicial del dataset.
2. Limpieza y preparación.
3. Entrenamiento de árbol.
4. Métricas.
5. Matriz de confusión.
6. Visualización del árbol.
7. Interpretación de reglas.
8. Conclusión profesional.

No se evalúa solo que el código funcione. Se evalúa la interpretación.

## Parte A — Comprensión del problema

Responda:

1. ¿Cuál es la pregunta de minería de datos?
2. ¿Cuál es la variable objetivo?
3. ¿Qué representa la clase 0?
4. ¿Qué representa la clase 1?
5. ¿Qué tipo de problema es?
6. ¿Qué variables podrían influir en la supervivencia?

## Parte B — Exploración

Realice:

1. Número de filas y columnas.
2. Valores faltantes.
3. Distribución de `Survived`.
4. Supervivencia por sexo.
5. Supervivencia por clase.
6. Una conclusión preliminar.

In [ ]:
# Parte B - Espacio de trabajo

print("Dimensiones:", df.shape)
print("\nValores faltantes:")
print(df.isna().sum())

print("\nDistribución de Survived:")
print(df["Survived"].value_counts(normalize=True))

## Parte C — Entrenamiento del modelo base

Entrene un árbol con:

- `criterion="gini"`
- `max_depth=3`
- `random_state=42`

Reporte:

1. Accuracy.
2. Precision.
3. Recall.
4. F1-score.
5. Matriz de confusión.

Explique cada resultado.

In [ ]:
# Parte C - Modelo base

modelo_taller = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    random_state=42
)

modelo_taller.fit(X_train, y_train)
pred_taller = modelo_taller.predict(X_test)

print(classification_report(y_test, pred_taller, target_names=["No sobrevivió", "Sobrevivió"]))

cm_taller = confusion_matrix(y_test, pred_taller)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_taller, display_labels=["No sobrevivió", "Sobrevivió"])
disp.plot()
plt.title("Matriz de confusión - Taller")
plt.show()

## Parte D — Comparación de profundidades

Entrene árboles con:

- `max_depth=1`
- `max_depth=2`
- `max_depth=3`
- `max_depth=5`
- `max_depth=8`
- `max_depth=None`

Construya una tabla con:

- Accuracy train.
- Accuracy test.
- F1 test.

Responda:

1. ¿Qué profundidad parece más adecuada?
2. ¿Hay evidencia de overfitting?
3. ¿Cuál modelo es más interpretable?
4. ¿Cuál elegiría para presentar?

In [ ]:
# Parte D - Comparación

profundidades_taller = [1, 2, 3, 5, 8, None]
resultados_taller = []

for p in profundidades_taller:
    modelo = DecisionTreeClassifier(
        criterion="gini",
        max_depth=p,
        random_state=42
    )

    modelo.fit(X_train, y_train)

    pred_train = modelo.predict(X_train)
    pred_test = modelo.predict(X_test)

    resultados_taller.append({
        "max_depth": str(p),
        "accuracy_train": accuracy_score(y_train, pred_train),
        "accuracy_test": accuracy_score(y_test, pred_test),
        "f1_test": f1_score(y_test, pred_test)
    })

pd.DataFrame(resultados_taller)

## Parte E — Interpretación de reglas

Visualice el árbol y responda:

1. ¿Cuál es la primera variable usada para dividir?
2. ¿Qué significa esa variable?
3. Escriba una regla del árbol en lenguaje natural.
4. ¿La regla parece coherente?
5. ¿Qué sesgos históricos podrían existir en este dataset?

In [ ]:
# Parte E - Visualización

plt.figure(figsize=(24, 12))

plot_tree(
    modelo_taller,
    feature_names=X.columns,
    class_names=["No sobrevivió", "Sobrevivió"],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Árbol de decisión del taller")
plt.show()

print(export_text(modelo_taller, feature_names=list(X.columns)))

## Parte F — Conclusión profesional

Redacte una conclusión de mínimo 10 líneas respondiendo:

1. ¿El árbol de decisión es adecuado para este problema?
2. ¿Qué métrica considera más importante y por qué?
3. ¿Qué variables parecen más relevantes?
4. ¿Qué limitaciones tiene el modelo?
5. ¿Qué haría antes de usarlo en un caso real?
6. ¿Qué otro algoritmo compararía después?

La conclusión debe sonar como un informe técnico.

# 25. Rúbrica sugerida

| Criterio | Puntaje |
|---|---:|
| Comprensión del problema | 0.6 |
| Exploración inicial | 0.7 |
| Preparación de datos | 0.7 |
| Entrenamiento del árbol | 0.8 |
| Evaluación con métricas | 0.9 |
| Comparación de profundidades | 0.8 |
| Interpretación de reglas | 0.3 |
| Conclusión profesional | 0.2 |
| **Total** | **5.0** |

---

# Cierre de clase

Los árboles de decisión permiten conectar:

1. Predicción.
2. Interpretabilidad.
3. Evaluación.
4. Reglas de decisión.
5. Control de sobreajuste.

La siguiente clase recomendada es:

## KNN y comparación de clasificadores

porque permite contrastar un modelo basado en reglas con un modelo basado en distancia.